# Round 12 | Build the latent representations

Run code cells individually with Shift+Enter, not Run All. Stop at the first failure or planned pause; save and bundle evidence. No automatic retries.

In [ ]:
from pathlib import Path
import json, sys, importlib.util
ROOT = Path.home() / "otto_feature_round12"
assert ROOT.is_dir(), ROOT
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
spec = importlib.util.spec_from_file_location("otto_launcher_12", ROOT / "launch.py")
launcher = importlib.util.module_from_spec(spec)
spec.loader.exec_module(launcher)
def stage(name):
    return launcher.run_stage(name)
def require(relative, key, expected):
    value = json.loads((ROOT / relative).read_text())
    assert value[key] == expected, (relative, value.get(key))
    print(expected)
    return value
print("KERNEL_READY")

## 1. Tests
This includes small synthetic factorization and native-ranker fixtures plus the installed-backend smoke. These tests have not been executed for this delivery.

In [ ]:
stage('tests')

## 2. Prepare sparse historical inputs
Reuses the certified existing history. No new data downloads, source Parquet scans, or graph rebuilding. Round12 additionally requires the completed Round11 report.

In [ ]:
stage('prepare')

## 3. Inspect the representation input gate

In [ ]:
inputs = require("outputs/representation_inputs/manifest.json", "status", "ROUND12_REPRESENTATION_INPUTS_READY")
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected+plotly_mimetype"
measures = {k: v for k,v in inputs.items() if k in ("vocabulary_size", "session_count", "nonzero_incidence", "pair_records", "source_history_events")}
fig = go.Figure(go.Bar(x=list(measures), y=list(measures.values())))
fig.update_layout(title="Historical input counts — different units, not model quality", yaxis_title="Count", xaxis_title="Input statistic")
fig.show()
print(json.dumps(inputs, indent=2))

## 4. Learn the frozen representations
Two SVD factorizations in Round11; six in Round12. These are counted compute, separate from the subsequent supervised ranker fits. An interrupted single SVD is not internally resumable; completed other units remain saved.

In [ ]:
stage('representations')

## 5. Inspect saved singular values
A low-rank historical basis is a feature construction result, not evidence of higher Recall@20.

In [ ]:
receipt = require("outputs/representations/manifest.json", "status", "ROUND12_REPRESENTATIONS_READY")
import numpy as np
fig = go.Figure()
for name in sorted(receipt["files"]):
    with np.load(ROOT / "outputs/representations" / name, allow_pickle=False) as values:
        singular = values["singular_values"]
    fig.add_trace(go.Scatter(x=list(range(1,len(singular)+1)), y=singular, mode="lines+markers", name=name))
fig.update_layout(title="Historical representation singular spectra", xaxis_title="Component", yaxis_title="Singular value")
fig.show()
print("Save this notebook before continuing.")